## Data loading and preparation

This notebook builds and evaluates text-based classifiers to predict the desired job position from Russian CVs, using BERT sentence embeddings and classical ML models. In this first section, we import dependencies and load the raw HeadHunter resumes dataset that will later be cleaned, filtered, and transformed into features and labels.

In [1]:
import sys
!"{sys.executable}" -m pip install fairlearn -q

In [2]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score

from fairlearn.metrics import (
    MetricFrame,
    demographic_parity_difference,
    demographic_parity_ratio,
    equalized_odds_difference,
    equalized_odds_ratio
)
from fairlearn.postprocessing import ThresholdOptimizer

RANDOM_STATE = 42

In [1]:
# for graphs
import sys
!"{sys.executable}" -m pip install matplotlib -q
import matplotlib.pyplot as plt
import numpy as np

plt.rcParams["figure.figsize"] = (8, 5)
plt.rcParams["axes.grid"] = True

In [3]:
# Attempt to use a local relative path, fallback to error if file not found
import os
import pandas as pd

possible_paths = [
    "data/raw/dst-3.0_16_1_hh_database.csv",      # relative to project root
    "./data/raw/dst-3.0_16_1_hh_database.csv",    # relative to CWD
    "../data/raw/dst-3.0_16_1_hh_database.csv",   # relative to possible 'notebooks' subdir
    "/data/raw/dst-3.0_16_1_hh_database.csv",     # absolute (original)
]

for path in possible_paths:
    if os.path.exists(path):
        data_path = path
        break
else:
    raise FileNotFoundError(
        "Data file dst-3.0_16_1_hh_database.csv not found in any of the expected locations: " +
        ", ".join(possible_paths)
    )
hh = pd.read_csv(data_path, sep=";")

print(hh.shape)
hh.head()

(44744, 12)


,"Пол, возраст",ЗП,Ищет работу на должность:,"Город, переезд, командировки",Занятость,График,Опыт работы,Последнее/нынешнее место работы,Последняя/нынешняя должность,Образование и ВУЗ,Обновление резюме,Авто
0,"Мужчина , 39 лет , родился 27 ноября 1979",29000 руб.,Системный администратор,"Советск (Калининградская область) , не готов к...","частичная занятость, проектная работа, полная ...","гибкий график, полный день, сменный график, ва...",Опыт работы 16 лет 10 месяцев Август 2010 — п...,"МАОУ ""СОШ № 1 г.Немана""",Системный администратор,Неоконченное высшее образование 2000 Балтийск...,16.04.2019 15:59,Имеется собственный автомобиль
1,"Мужчина , 60 лет , родился 20 марта 1959",40000 руб.,Технический писатель,"Королев , не готов к переезду , готов к редким...","частичная занятость, проектная работа, полная ...","гибкий график, полный день, сменный график, уд...",Опыт работы 19 лет 5 месяцев Январь 2000 — по...,Временный трудовой коллектив,"Менеджер проекта, Аналитик, Технический писатель",Высшее образование 1981 Военно-космическая ак...,12.04.2019 08:42,Не указано
2,"Женщина , 36 лет , родилась 12 августа 1982",20000 руб.,Оператор,"Тверь , не готова к переезду , не готова к ком...",полная занятость,полный день,Опыт работы 10 лет 3 месяца Октябрь 2004 — Де...,ПАО Сбербанк,Кассир-операционист,Среднее специальное образование 2002 Профессио...,16.04.2019 08:35,Не указано
3,"Мужчина , 38 лет , родился 25 июня 1980",100000 руб.,Веб-разработчик (HTML / CSS / JS / PHP / базы ...,"Саратов , не готов к переезду , готов к редким...","частичная занятость, проектная работа, полная ...","гибкий график, удаленная работа",Опыт работы 18 лет 9 месяцев Август 2017 — Ап...,OpenSoft,Инженер-программист,Высшее образование 2002 Саратовский государст...,08.04.2019 14:23,Не указано
4,"Женщина , 26 лет , родилась 3 марта 1993",140000 руб.,Региональный менеджер по продажам,"Москва , не готова к переезду , готова к коман...",полная занятость,полный день,Опыт работы 5 лет 7 месяцев Региональный мене...,Мармелад,Менеджер по продажам,Высшее образование 2015 Кгу Психологии и педаг...,22.04.2019 10:32,Не указано


## Demographic and geographic feature engineering

In this part, we clean the raw location and personal information from the resumes and turn it into structured features. We extract the city from the `"Город, переезд, командировки"` field, drop cities with too few samples, and derive gender and age groups from the `"Пол, возраст"` field so they can later be used as sensitive attributes for fairness analysis and stratification.

In [4]:
import pandas as pd
import numpy as np

# extract city name
def extract_city(s):
    if pd.isna(s):
        return "Unknown"
    city = str(s).split(",")[0].strip()
    city = city.replace("г.", "").strip()
    return city

hh["city"] = hh["Город, переезд, командировки"].apply(extract_city)

print("All cities:")
city_counts = hh["city"].value_counts()
print(city_counts)
print(f"\nNumber of city value lines: {len(city_counts)}")

All cities:
city
Москва                                  16621
Санкт-Петербург                          4937
Краснодар                                1066
Новосибирск                               958
Казань                                    872
                                        ...  
Краснослободск (Республика Мордовия)        1
Холмск                                      1
Магас                                       1
Октябрьская                                 1
Темрюк                                      1
Name: count, Length: 985, dtype: int64

Number of city value lines: 985


In [5]:
MIN_SAMPLES_PER_CITY = 300

valid_cities = city_counts[city_counts >= MIN_SAMPLES_PER_CITY].index

hh_filtered = hh[hh["city"].isin(valid_cities)].copy()

print("Cities after filtering:", len(valid_cities))
print("Dataset size after filtering:", hh_filtered.shape)

display(hh_filtered["city"].value_counts().head(30))

Cities after filtering: 18
Dataset size after filtering: (30948, 13)


city
Москва             16621
Санкт-Петербург     4937
Краснодар           1066
Новосибирск          958
Казань               872
Екатеринбург         734
Самара               703
Ростов-на-Дону       607
Нижний Новгород      598
Уфа                  565
Алматы               556
Воронеж              538
Пермь                445
Красноярск           407
Тюмень               355
Минск                353
Челябинск            330
Омск                 303
Name: count, dtype: int64

In [6]:
import re
import numpy as np
import pandas as pd

# extract gender
def extract_gender(s):
    if pd.isna(s):
        return "Unknown"
    s = str(s).lower()
    if "муж" in s:
        return "Male"
    elif "жен" in s:
        return "Female"
    else:
        return "Unknown"

# extract age
def extract_age(s):
    if pd.isna(s):
        return np.nan
    match = re.search(r'(\d+)\s*лет', str(s))
    if match:
        return int(match.group(1))
    return np.nan

# apply to column
hh_filtered["gender"] = hh_filtered["Пол, возраст"].apply(extract_gender)
hh_filtered["age"] = hh_filtered["Пол, возраст"].apply(extract_age)

# create age groups
def age_to_group(age):
    if pd.isna(age):
        return "Unknown"
    age = int(age)
    if age < 18:
        return "<18"
    elif age <= 21:
        return "18–21"
    elif age <= 25:
        return "22–25"
    elif age <= 35:
        return "26–35"
    elif age <= 50:
        return "36–50"
    else:
        return "50+"

hh_filtered["age_group"] = hh_filtered["age"].apply(age_to_group)
age_order = ["<18", "18–21", "22–25", "26–35", "36–50", "50+", "Unknown"]
hh_filtered["age_group"] = pd.Categorical(
    hh_filtered["age_group"],
    categories=age_order,
    ordered=True
)

# print distributions
print("Gender distribution (with Unknown):")
gender_counts = hh_filtered["gender"].value_counts(dropna=False)
print(gender_counts)

print("\nShare by gender:")
print((gender_counts / gender_counts.sum()).round(3))

print("\nAge group distribution (ordered):")
print(hh_filtered["age_group"].value_counts(sort=False))

Gender distribution (with Unknown):
gender
Male      24867
Female     6081
Name: count, dtype: int64

Share by gender:
gender
Male      0.804
Female    0.196
Name: count, dtype: float64

Age group distribution (ordered):
age_group
<18           12
18–21        454
22–25       1560
26–35      10376
36–50       5419
50+          450
Unknown    12677
Name: count, dtype: int64


## Building text representations and multiclass labels

Here we construct a single free-text field `resume_text` by concatenating job experience, current/last position, company, and education fields. We also define the target label as the profession the candidate is looking for, filter out very rare professions, and split the resulting dataset into train/validation/test sets for a multiclass classification problem.

In [7]:
# 1. Собираем текст резюме
text_columns = [
    "Опыт работы",
    "Последняя/нынешняя должность",
    "Последнее/нынешнее место работы",
    "Образование и ВУЗ"
]

for col in text_columns:
    hh_filtered[col] = hh_filtered[col].fillna("")

hh_filtered["resume_text"] = (
    hh_filtered["Опыт работы"] + " " +
    hh_filtered["Последняя/нынешняя должность"] + " " +
    hh_filtered["Последнее/нынешнее место работы"] + " " +
    hh_filtered["Образование и ВУЗ"]
)

# 2. Метка = профессия
hh_filtered["label"] = (
    hh_filtered["Ищет работу на должность:"]
    .astype(str)
    .str.strip()
)

# убираем пустые значения
hh_filtered = hh_filtered[hh_filtered["label"] != ""]
hh_filtered = hh_filtered[hh_filtered["label"] != "nan"]

print("Dataset shape before profession filter:", hh_filtered.shape)
print("Total professions:", hh_filtered["label"].nunique())

Dataset shape before profession filter: (30948, 18)
Total professions: 11040


In [8]:
from sklearn.model_selection import train_test_split

# 1. Фильтрация редких профессий
label_counts = hh_filtered["label"].value_counts()

MIN_SAMPLES_PER_PROFESSION = 50
valid_labels = label_counts[label_counts >= MIN_SAMPLES_PER_PROFESSION].index

hh_filtered = hh_filtered[hh_filtered["label"].isin(valid_labels)].copy()

print("Professions after filter:", len(valid_labels))
print("Dataset size after filter:", hh_filtered.shape)

# 2. Финальный датафрейм (создаём ПОСЛЕ фильтрации)
df = hh_filtered[[
    "resume_text",
    "label",
    "city",
    "gender",
    "age_group"
]].dropna()

print("Final dataset shape:", df.shape)
print("Unique professions:", df["label"].nunique())

# 3. Разбиение на train / val / test
X = df["resume_text"].values
y = df["label"].values

X_temp, X_test, y_temp, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

X_train, X_val, y_train, y_val = train_test_split(
    X_temp,
    y_temp,
    test_size=0.25,
    random_state=42,
    stratify=y_temp
)

print("Train:", len(X_train))
print("Val:", len(X_val))
print("Test:", len(X_test))

Professions after filter: 73
Dataset size after filter: (12718, 18)
Final dataset shape: (12718, 5)
Unique professions: 73
Train: 7630
Val: 2544
Test: 2544


## Sentence-transformer embeddings and baseline classifier

In this section we download a pre-trained sentence-transformer model (`paraphrase-MiniLM-L3-v2`) from Hugging Face and use it to encode each resume into a dense vector representation. On top of these fixed embeddings, we first train a simple logistic regression classifier and evaluate its accuracy and macro F1 on the test set as an initial baseline across many profession classes.

In [9]:
pip install sentence-transformers

Note: you may need to restart the kernel to use updated packages.


In [10]:
import requests
print(requests.get("https://huggingface.co").status_code)

/Users/natashaagapova/Documents/! INNOPOLIS/! THESIS/my-repository/.venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


200


In [12]:
from huggingface_hub import snapshot_download

snapshot_download(
    repo_id="sentence-transformers/paraphrase-MiniLM-L3-v2",
    local_dir="./local_bert",
    local_dir_use_symlinks=False
)

/Users/natashaagapova/Documents/! INNOPOLIS/! THESIS/my-repository/.venv/lib/python3.9/site-packages/huggingface_hub/file_download.py:986: UserWarning: `local_dir_use_symlinks` parameter is deprecated and will be ignored. The process to download files to a local folder has been updated and do not rely on symlinks anymore. You only need to pass a destination folder as`local_dir`.
For more details, check out https://huggingface.co/docs/huggingface_hub/main/en/guides/download#download-files-to-local-folder.
  warnings.warn(
Fetching 27 files: 100%|██████████| 27/27 [07:07<00:00, 15.85s/it]


'/Users/natashaagapova/Documents/! INNOPOLIS/! THESIS/my-repository/notebooks/local_bert'

In [13]:
bert = SentenceTransformer("./local_bert")

In [14]:
X_train_emb = bert.encode(X_train, batch_size=32, show_progress_bar=True)
X_val_emb   = bert.encode(X_val, batch_size=32, show_progress_bar=True)
X_test_emb  = bert.encode(X_test, batch_size=32, show_progress_bar=True)

print("Train:", X_train_emb.shape)
print("Val:", X_val_emb.shape)
print("Test:", X_test_emb.shape)

Batches: 100%|██████████| 80/80 [00:05<00:00, 13.43it/s]

Train: (7630, 384)
Val: (2544, 384)
Test: (2544, 384)


In [15]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score

# classifier
clf = LogisticRegression(max_iter=2000)
clf.fit(X_train_emb, y_train)

# predictions
y_test_pred = clf.predict(X_test_emb)

# metrics
acc = accuracy_score(y_test, y_test_pred)
f1 = f1_score(y_test, y_test_pred, average="macro")

print("Accuracy:", acc)
print("F1 (macro):", f1)

Accuracy: 0.392688679245283
F1 (macro): 0.26238183462970943


## Focusing on frequent professions and stronger classifier

Finally, we restrict the problem to a subset of more frequent professions to reduce label noise and class sparsity, rebuild the dataset, and re-encode resumes with the same sentence-transformer. We then train a stronger linear SVM classifier on this reduced label space and compare its performance (accuracy and macro F1) to the previous logistic regression baseline to understand how class filtering and model choice affect results.

In [17]:
# Ужесточаем порог для профессий
label_counts = hh_filtered["label"].value_counts()

MIN_SAMPLES_PER_PROFESSION = 50
valid_labels = label_counts[label_counts >= MIN_SAMPLES_PER_PROFESSION].index

hh_strong = hh_filtered[hh_filtered["label"].isin(valid_labels)].copy()

print("Classes:", hh_strong["label"].nunique())
print("Rows:", hh_strong.shape[0])
print(hh_strong["label"].value_counts().head(20))


Classes: 25
Rows: 9164
label
Системный администратор             1717
Аналитик                             664
Менеджер проектов                    609
Инженер                              553
Руководитель проекта                 552
Руководитель проектов                543
Специалист технической поддержки     473
Менеджер интернет-магазина           372
Менеджер проекта                     365
Программист                          294
Технический специалист               285
Специалист по IT                     237
Интернет-маркетолог                  235
Менеджер                             234
Инженер-программист                  233
Специалист                           208
Программист-разработчик              196
Сервисный инженер                    195
Менеджер по продажам                 191
Инженер технической поддержки        183
Name: count, dtype: int64


In [18]:
from sklearn.model_selection import train_test_split

df_strong = hh_strong[[
    "resume_text",
    "label",
    "city",
    "gender",
    "age_group"
]].dropna()

print("Final strong dataset:", df_strong.shape)
print("Unique professions:", df_strong["label"].nunique())

X = df_strong["resume_text"].values
y = df_strong["label"].values

X_temp, X_test, y_temp, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

X_train, X_val, y_train, y_val = train_test_split(
    X_temp, y_temp, test_size=0.25, random_state=42, stratify=y_temp
)

print("Train:", len(X_train))
print("Val:", len(X_val))
print("Test:", len(X_test))

Final strong dataset: (9164, 5)
Unique professions: 25
Train: 5498
Val: 1833
Test: 1833


In [19]:
X_train_emb = bert.encode(X_train, batch_size=32, show_progress_bar=True)
X_val_emb   = bert.encode(X_val, batch_size=32, show_progress_bar=True)
X_test_emb  = bert.encode(X_test, batch_size=32, show_progress_bar=True)

print("Train:", X_train_emb.shape)
print("Val:", X_val_emb.shape)
print("Test:", X_test_emb.shape)


Batches: 100%|██████████| 58/58 [00:04<00:00, 12.90it/s]

Train: (5498, 384)
Val: (1833, 384)
Test: (1833, 384)


In [20]:
from sklearn.svm import LinearSVC
from sklearn.metrics import accuracy_score, f1_score

svm = LinearSVC()
svm.fit(X_train_emb, y_train)

y_test_pred = svm.predict(X_test_emb)

print("Accuracy:", accuracy_score(y_test, y_test_pred))
print("F1 (macro):", f1_score(y_test, y_test_pred, average="macro"))

Accuracy: 0.5280960174577196
F1 (macro): 0.4943651785880543


In [21]:
# create test dataframe
test_df = df_strong.copy()

# take only rows that fell into test
test_df = test_df.iloc[-len(y_test):].copy()

test_df["y_true"] = y_test
test_df["y_pred"] = y_test_pred

print(test_df.head())

                                             resume_text  \
35877  Опыт работы 11 лет 5 месяцев  Февраль 2013 — п...   
35878  Опыт работы 2 года  Менеджер 50 000 руб. Инфор...   
35879  Опыт работы 15 лет 11 месяцев  Август 2017 — Д...   
35880  Опыт работы 6 лет 9 месяцев  IT-специалист 50 ...   
35885  Опыт работы 7 лет  Декабрь 2015 — по настоящее...   

                       label             city gender age_group  \
35877  Руководитель проектов  Санкт-Петербург   Male   Unknown   
35878               Менеджер          Воронеж   Male   Unknown   
35879  Руководитель проектов           Москва   Male     36–50   
35880          IT-специалист           Москва   Male     26–35   
35885   Frontend-разработчик   Ростов-на-Дону   Male     26–35   

                                 y_true                      y_pred  
35877                  Специалист по IT            Специалист по IT  
35878  Специалист технической поддержки  Менеджер интернет-магазина  
35879                          А

In [24]:
from sklearn.metrics import accuracy_score

def group_accuracy(df, group_col):
    results = []
    for group in sorted(df[group_col].dropna().unique()):
        group_df = df[df[group_col] == group]
        acc = accuracy_score(group_df["y_true"], group_df["y_pred"])
        results.append((group, len(group_df), round(acc, 3)))
    return results

In [25]:
# conclusion on every sensitive attribut

print("Accuracy by city:")
for row in group_accuracy(test_df, "city"):
    print(row)

print("\nAccuracy by gender:")
for row in group_accuracy(test_df, "gender"):
    print(row)

print("\nAccuracy by age group:")
for row in group_accuracy(test_df, "age_group"):
    print(row)

Accuracy by city:
('Алматы', 48, 0.479)
('Воронеж', 34, 0.529)
('Екатеринбург', 58, 0.448)
('Казань', 55, 0.582)
('Краснодар', 61, 0.574)
('Красноярск', 29, 0.483)
('Минск', 20, 0.5)
('Москва', 921, 0.519)
('Нижний Новгород', 41, 0.537)
('Новосибирск', 60, 0.55)
('Омск', 15, 0.733)
('Пермь', 37, 0.649)
('Ростов-на-Дону', 26, 0.538)
('Самара', 51, 0.49)
('Санкт-Петербург', 295, 0.549)
('Тюмень', 19, 0.632)
('Уфа', 39, 0.462)
('Челябинск', 24, 0.458)

Accuracy by gender:
('Female', 367, 0.569)
('Male', 1466, 0.518)

Accuracy by age group:
('18–21', 32, 0.562)
('22–25', 122, 0.5)
('26–35', 688, 0.515)
('36–50', 248, 0.573)
('50+', 15, 0.533)
('<18', 2, 0.5)
('Unknown', 726, 0.529)
